In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("eval_data.csv")
ds_eval = data.copy()

dl_eval = DF_Batcher(ds_eval, batch_size=200, shuffle=False)

In [ ]:
data

In [ ]:
from src.eval import (
    HarmBenchEvaluator,
    LlamaEvaluator,
    TemplateEvaluator,
    BeaverEvaluator,
    Evaluator,
    LlamaGuardEvaluator,
    MultiEvaluator,
)

from gserve.configs import ServeConfig, LLMConfig
import os


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     use_context=False,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    # ),
    MultiEvaluator(
        evaluators=[
            BeaverEvaluator(device_map="cuda:0"),
            LlamaEvaluator(
                model_name="meta-llama/Llama-3.1-8B-Instruct",
                serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
            ),
        ],
        combine_fn=lambda res: 1.0 if (res["Beaver/Thresh@0.0"] >= 0.0 and res["Llama-3.1-8B-Instruct"] >= 1.0) else 0.0,
    )
]

In [ ]:
eval_results = {}

for ev in evaluators:
    print(f"Running evaluator: {ev.name}")
    results = ev.evaluate(dl_eval)
    eval_results.update(results)
    print(f"Results: {results}")

In [ ]:
ds_eval

In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)